# Regressão Feed-Forward para Previsão de Log-Retorno

**Disciplina:** Projeto de Deep Learning

**Curso:** Ciência de Dados e IA

**Professor(a):** Renan Santos Mendes

**Email:** renansantosmendes@gmail.com

Este notebook treina um modelo de regressão feed-forward para prever o próximo log-retorno diário de uma ação a partir de uma janela de log-retornos passados. Ele reaproveita o dataset de reconstrução, o wrapper de tracking e as utilidades de modelo, normalização e coleta de dados já implementadas em `src/generative_models/`.

A célula abaixo importa as bibliotecas padrão usadas para manipulação de dados, plotagem e treinamento, junto com as classes e funções próprias do projeto.

In [ ]:
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchinfo import summary
from tqdm.auto import tqdm

from pgl_utils.deep_learning import (
    plot_loss_curve,
    plot_time_series,
    plot_two_series_comparison,
)

from generative_models.common.normalization import normalize_series
from generative_models.data.datasets import SlidingWindowReconstructionDataset
from generative_models.data.market_data import (
    compute_log_return_values,
    fetch_price_series,
)
from generative_models.models.regression.feed_forward_regression_model import (
    FeedForwardRegressionModel,
)
from generative_models.tracking.wandb_tracker import WandbExperimentTracker

In [ ]:
from dotenv import load_dotenv

load_dotenv()

As constantes abaixo controlam o experimento (ticker, intervalo de datas, semente de aleatoriedade) e são usadas em todo o notebook, inclusive na configuração enviada ao `wandb`, para que toda a execução permaneça reprodutível a partir de uma única fonte de verdade.

In [ ]:
RANDOM_SEED = 42
TICKER_SYMBOL = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2026-01-01"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Ticker: {TICKER_SYMBOL}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Random seed: {RANDOM_SEED}")

Os preços de fechamento diários são coletados com `fetch_price_series`, que encapsula o `yfinance` e retorna um data frame de uma única coluna.

In [ ]:
price_data = fetch_price_series(TICKER_SYMBOL, START_DATE, END_DATE)
# price_data.head()

A série completa de preços de fechamento é plotada antes de qualquer transformação ser aplicada.

In [ ]:
price_figure = plot_time_series(
    series_data=price_data,
    column_name="close_price",
    chart_title=f"{TICKER_SYMBOL} - Closing Price",
)
price_figure.show()

O modelo é treinado para prever o log-retorno do dia seguinte em vez do preço bruto. Um modelo treinado diretamente sobre o preço pode alcançar um MSE muito baixo apenas copiando o último valor conhecido, sem aprender nenhuma dinâmica temporal real. A série de log-retornos é praticamente estacionária, então o modelo não tem mais esse atalho trivial e precisa encontrar um sinal preditivo de fato, se ele existir. `compute_log_return_values` calcula `log(price[t]) - log(price[t - 1])` a partir da série de preços.

In [ ]:
price_values = price_data["close_price"].to_numpy(dtype=np.float32)
log_return_values = compute_log_return_values(price_values)
log_return_tensor = torch.from_numpy(log_return_values)

train_split_index = int(len(log_return_tensor) * 0.8)

train_log_return_tensor = log_return_tensor[:train_split_index]
validation_log_return_tensor = log_return_tensor[train_split_index:]

normalized_train_return_tensor, return_train_mean, return_train_std = (
    normalize_series(train_log_return_tensor)
)
normalized_validation_return_tensor = (
    validation_log_return_tensor - return_train_mean
) / return_train_std

`SlidingWindowReconstructionDataset` gera uma janela por amostra, sem separação entre entrada e saída. Para reaproveitá-lo na regressão do próximo valor, cada janela é construída com `regression_window_size + 1` valores: os primeiros `regression_window_size` valores se tornam a entrada do modelo, e o último valor se torna o alvo da previsão. Essa separação é aplicada uma vez por batch, logo depois que o `DataLoader` monta as janelas.

In [ ]:
regression_window_size = 30
regression_batch_size = 32

train_regression_dataset = SlidingWindowReconstructionDataset(
    series_values=normalized_train_return_tensor.numpy(),
    window_size=regression_window_size + 1,
)
validation_regression_dataset = SlidingWindowReconstructionDataset(
    series_values=normalized_validation_return_tensor.numpy(),
    window_size=regression_window_size + 1,
)

dataloader_generator = torch.Generator()
dataloader_generator.manual_seed(RANDOM_SEED)

train_regression_dataloader = DataLoader(
    train_regression_dataset,
    batch_size=regression_batch_size,
    shuffle=True,
    generator=dataloader_generator,
)
validation_regression_dataloader = DataLoader(
    validation_regression_dataset,
    batch_size=regression_batch_size,
    shuffle=False,
)

print(len(train_regression_dataset), len(validation_regression_dataset))

`FeedForwardRegressionModel` empilha duas camadas ocultas com `ReLU`, seguidas por uma camada de saída linear, permitindo aprender combinações não-lineares da janela de entrada em vez de uma regressão linear simples.

In [ ]:
hidden_layer_size = 64

regression_model = FeedForwardRegressionModel(
    input_window_size=regression_window_size,
    hidden_layer_size=hidden_layer_size,
)
print(regression_model)

summary(
    regression_model,
    input_size=(regression_batch_size, regression_window_size),
    device="cpu",
)

`WandbExperimentTracker` centraliza a tentativa de login, a inicialização do run e o registro de métricas. A `WANDB_API_KEY` é lida de um arquivo `.env` local (veja `.env.example`); se nenhuma chave for encontrada, um run anônimo é iniciado. O laço de treinamento abaixo otimiza o erro quadrático médio (MSE) entre o log-retorno previsto e o real, usando o otimizador Adam.

In [ ]:
number_of_epochs = 100
learning_rate = 0.001

experiment_tracker = WandbExperimentTracker(
    project_name="pytorch-tensores-log-return",
    config={
        "ticker": TICKER_SYMBOL,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "random_seed": RANDOM_SEED,
        "epochs": number_of_epochs,
        "learning_rate": learning_rate,
        "window_size": regression_window_size,
        "batch_size": regression_batch_size,
        "hidden_layer_size": hidden_layer_size,
        "optimizer": "Adam",
        "loss_function": "MSE",
        "target": "log_return",
    },
)

In [ ]:

experiment_tracker.start_run()

loss_function = nn.MSELoss()
optimizer = optim.Adam(regression_model.parameters(), lr=learning_rate)

training_loss_history = []
epoch_progress_bar = tqdm(range(number_of_epochs), desc="Training", unit="epoch")

for epoch_index in epoch_progress_bar:
    epoch_loss_total = 0.0

    for batch_windows in train_regression_dataloader:
        batch_input = batch_windows[:, :-1]
        batch_target = batch_windows[:, -1:]

        optimizer.zero_grad()
        predicted_values = regression_model(batch_input)
        loss_value = loss_function(predicted_values, batch_target)
        loss_value.backward()
        optimizer.step()
        epoch_loss_total += loss_value.item() * batch_input.shape[0]

    epoch_average_loss = epoch_loss_total / len(train_regression_dataset)
    training_loss_history.append(epoch_average_loss)

    experiment_tracker.log_metrics(
        {"epoch": epoch_index + 1, "train/loss_mse": epoch_average_loss}
    )
    epoch_progress_bar.set_postfix(loss=f"{epoch_average_loss:.6f}")

experiment_tracker.log_model(
    model=regression_model,
    model_name="feed-forward-regression-model",
    model_file_path="regression_model.pt",
    metadata={"hidden_layer_size": hidden_layer_size},
)
experiment_tracker.finish_run()

training_loss_figure = plot_loss_curve(
    training_loss_history,
    "Learning Curve - Log-Return Prediction",
    loss_series_name="Train MSE (log-return)",
)
training_loss_figure.show()

O conjunto de validação é avaliado sem calcular gradientes. As previsões e os alvos são desnormalizados usando `return_train_mean` e `return_train_std`, calculados apenas a partir do conjunto de treino. Como verificação de sanidade, o MSE do modelo é comparado com a baseline ingênua "log-retorno previsto = 0" (equivalente a "o preço de amanhã é igual ao de hoje"). Não superar essa baseline é esperado para uma ação líquida como a `AAPL`, sob a hipótese de mercado eficiente.

In [ ]:
regression_model.eval()
validation_predictions = []
validation_targets = []

with torch.no_grad():
    for batch_windows in validation_regression_dataloader:
        batch_input = batch_windows[:, :-1]
        batch_target = batch_windows[:, -1:]

        predicted_values = regression_model(batch_input)
        validation_predictions.append(predicted_values)
        validation_targets.append(batch_target)

validation_predictions_tensor = torch.cat(validation_predictions, dim=0).squeeze(1)
validation_targets_tensor = torch.cat(validation_targets, dim=0).squeeze(1)

predicted_returns = (
    validation_predictions_tensor.numpy() * return_train_std + return_train_mean
)
actual_returns = (
    validation_targets_tensor.numpy() * return_train_std + return_train_mean
)

model_mse = np.mean((actual_returns - predicted_returns) ** 2)
naive_mse = np.mean(actual_returns ** 2)

print(f"Model MSE (log-return):    {model_mse:.6f}")
print(f"Naive baseline MSE (r=0):  {naive_mse:.6f}")

Os log-retornos reais e previstos no conjunto de validação são plotados lado a lado.

In [ ]:
validation_time_indices = np.arange(len(predicted_returns))

returns_figure = plot_two_series_comparison(
    x_axis_values=validation_time_indices,
    first_series=actual_returns,
    second_series=predicted_returns,
    first_series_name="Actual log-return",
    second_series_name="Predicted log-return",
    chart_title=f"{TICKER_SYMBOL} - Neural Network: Actual vs Predicted Log-Return (validation)",
)
returns_figure.show()

Por fim, o preço é reconstruído a partir do log-retorno previsto, apenas para fins de visualização: `predicted_price[t + 1] = price[t] * exp(predicted_return[t + 1])`. Essa reconstrução pode parecer visualmente boa mesmo quando o modelo não aprendeu nada além do trivial, então a métrica que realmente decide se o modelo é útil é o MSE calculado no espaço de retorno acima, não a aparência do gráfico de preços.

In [ ]:
validation_start_price_indices = (
    train_split_index + regression_window_size + np.arange(len(predicted_returns))
)
previous_known_prices = price_values[validation_start_price_indices]

reconstructed_predicted_prices = previous_known_prices * np.exp(predicted_returns)
reconstructed_actual_prices = previous_known_prices * np.exp(actual_returns)

price_reconstruction_figure = plot_two_series_comparison(
    x_axis_values=validation_time_indices,
    first_series=reconstructed_actual_prices,
    second_series=reconstructed_predicted_prices,
    first_series_name="Actual price",
    second_series_name="Price reconstructed from predicted log-return",
    chart_title=f"{TICKER_SYMBOL} - Price Reconstructed from Predicted Log-Return (validation)",
)
price_reconstruction_figure.show()